In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Partie I : Classification avec MLP sur Breast Cancer Wisconsin
## 1. Préparation des données
Nous utilisons le dataset Breast Cancer. Les données sont séparées en apprentissage (70%), validation (15%) et test (15%), puis normalisées avec `StandardScaler` pour faciliter l'apprentissage de notre réseau de neurones.

In [ ]:
# 1. Chargement du dataset Breast Cancer Wisconsin
data = load_breast_cancer()
X = data.data
y = data.target

# 2. Séparation : 70% Train, 15% Validation, 15% Test
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.176, random_state=42)

# 3. Normalisation
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# 4. Conversion en tenseurs PyTorch
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 5. Création des DataLoaders (mini-lots)
batch_size = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)

print(f"Dimensions d'entrée : {X_train.shape[1]} features.")

## 2. Implémentation des modèles
Pour ce projet, nous implémentons le perceptron multicouche de deux manières différentes :
* **nn.Sequential** : Une approche séquentielle simple qui enchaîne les couches les unes après les autres.
* **Classe personnalisée (nn.Module)** : Une approche plus flexible permettant un contrôle total sur le flux des données dans la méthode `forward()`. C'est cette version que nous utiliserons pour l'entraînement.

In [ ]:
# Version Sequential
mlp_sequential = nn.Sequential(
    nn.LazyLinear(64),
    nn.ReLU(),
    nn.LazyLinear(32),
    nn.ReLU(),
    nn.LazyLinear(1)
)

# Version Classe Personnalisée
class MLPCustom(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.LazyLinear(64)
        self.hidden2 = nn.LazyLinear(32)
        self.output = nn.LazyLinear(1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        return self.output(x)

mlp_custom = MLPCustom()

# Initialisation des Lazy layers en passant un lot factice
dummy_input = X_train_t[:5]
_ = mlp_sequential(dummy_input)
_ = mlp_custom(dummy_input)

print("Modèles initialisés avec succès !")

## 3. Inspection des paramètres et Initialisation
Il est crucial de comprendre la structure interne de notre modèle. Nous utilisons `named_parameters()` pour vérifier les dimensions de chaque couche, et `state_dict()` pour accéder aux tenseurs des poids et biais.

Ensuite, nous définissons trois stratégies d'initialisation :
* **Gaussienne** : Initialise les poids avec une distribution normale (moyenne 0, faible écart-type).
* **Constante** : Initialise tous les poids à 1 (déconseillé en pratique à cause de la symétrie).
* **Xavier (Glorot)** : Stabilise la variance des activations. C'est la méthode que nous appliquerons à notre modèle pour un apprentissage optimal.

Enfin, nous déplaçons notre modèle sur le GPU (si disponible) pour accélérer les calculs.

In [ ]:
# --- 1. Inspection des paramètres ---
print("=== Structure via named_parameters() ===")
for name, param in mlp_custom.named_parameters():
    print(f"Nom : {name} | Dimension : {param.shape} | Requiert gradient : {param.requires_grad}")

print("\n=== Aperçu via state_dict() ===")
print(f"Clés disponibles dans le state_dict : {list(mlp_custom.state_dict().keys())}")

# --- 2. Fonctions d'initialisation ---
def init_gaussienne(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0, std=0.01)
        nn.init.zeros_(module.bias)

def init_constante(module):
    if isinstance(module, nn.Linear):
        nn.init.constant_(module.weight, 1.0)
        nn.init.zeros_(module.bias)

def init_xavier(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        nn.init.zeros_(module.bias)

# Application de l'initialisation de Xavier
mlp_custom.apply(init_xavier)
print("\nInitialisation de Xavier appliquée avec succès !")

# --- 3. Gestion du GPU (Device) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlp_custom = mlp_custom.to(device)
print(f"\nModèle déplacé sur le device : {device}")

## 4. Entraînement du Modèle
Pour entraîner notre MLP sur cette tâche de classification binaire, nous définissons :
* **Fonction de perte** : `BCEWithLogitsLoss`. C'est la version la plus stable en PyTorch car elle combine une activation Sigmoïde et la perte d'entropie croisée binaire (Binary Cross Entropy) en une seule opération.
* **Optimiseur** : `Adam`, très performant pour ajuster le taux d'apprentissage de chaque paramètre dynamiquement.

À chaque itération (epoch), le modèle s'entraîne sur les mini-lots (batches) d'apprentissage, met à jour ses poids via la rétropropagation (`loss.backward()`), puis est évalué sur l'ensemble de validation avec `torch.no_grad()` pour désactiver le calcul des gradients et économiser la mémoire.

In [ ]:
import torch.optim as optim

# --- 1. Hyperparamètres ---
epochs = 50
learning_rate = 0.001

# Fonction de perte et optimiseur
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(mlp_custom.parameters(), lr=learning_rate)

# Listes pour stocker l'historique (très utile pour tes futures courbes)
train_losses = []
val_losses = []
val_accuracies = []

print(" Début de l'entraînement sur GPU...\n")

# --- 2. Boucle d'entraînement ---
for epoch in range(epochs):
    
    # -- Phase d'apprentissage --
    mlp_custom.train() # Passe le modèle en mode entraînement
    epoch_train_loss = 0
    
    for X_batch, y_batch in train_loader:
        # Transfert des données sur le GPU
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # Propagation avant (Forward)
        predictions = mlp_custom(X_batch)
        loss = criterion(predictions, y_batch)
        
        # Rétropropagation (Backward) et mise à jour des poids
        optimizer.zero_grad() # Réinitialise les gradients
        loss.backward()       # Calcule les nouveaux gradients
        optimizer.step()      # Met à jour les paramètres
        
        epoch_train_loss += loss.item()
        
    # Moyenne de la perte sur tous les lots
    train_losses.append(epoch_train_loss / len(train_loader))
    
    # -- Phase de validation --
    mlp_custom.eval() # Passe le modèle en mode évaluation
    epoch_val_loss = 0
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad(): # Désactive le calcul des gradients (gain de temps/mémoire)
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            predictions = mlp_custom(X_batch)
            loss = criterion(predictions, y_batch)
            epoch_val_loss += loss.item()
            
            # Calcul de la précision (Accuracy)
            # Les prédictions > 0 correspondent à la classe 1 (BCEWithLogitsLoss)
            predicted_classes = (predictions > 0).float()
            correct_predictions += (predicted_classes == y_batch).sum().item()
            total_samples += y_batch.size(0)
            
    val_losses.append(epoch_val_loss / len(val_loader))
    val_accuracy = correct_predictions / total_samples
    val_accuracies.append(val_accuracy)
    
    # Affichage des résultats tous les 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"Train Loss: {train_losses[-1]:.4f} | "
              f"Val Loss: {val_losses[-1]:.4f} | "
              f"Val Accuracy: {val_accuracy*100:.2f}%")

print("\n Entraînement terminé !")

## 5. Évaluation sur l'ensemble de Test et Courbes d'apprentissage
Pour valider définitivement notre modèle, nous l'évaluons sur l'ensemble de test (des données qu'il n'a jamais vues). Nous utilisons des métriques adaptées à la classification médicale :
* **Précision (Precision) :** Sur toutes les tumeurs prédites malignes, combien l'étaient réellement ?
* **Rappel (Recall) :** Sur toutes les tumeurs réellement malignes, combien ont été détectées ? (Crucial en médecine).
* **F1-score :** La moyenne harmonique de la précision et du rappel.
* **Matrice de confusion :** Pour visualiser les vrais/faux positifs et négatifs.

Enfin, nous traçons les courbes d'apprentissage pour vérifier la convergence.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# --- 1. Tracé des courbes d'apprentissage ---
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', color='blue')
plt.plot(val_losses, label='Validation Loss', color='orange')
plt.title('Évolution de la fonction de perte (Loss)')
plt.xlabel('Époques')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label='Validation Accuracy', color='green')
plt.title('Évolution de la précision (Accuracy)')
plt.xlabel('Époques')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# --- 2. Évaluation sur l'ensemble de Test ---
mlp_custom.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        predictions = mlp_custom(X_batch)
        predicted_classes = (predictions > 0).float().cpu().numpy()
        
        y_pred.extend(predicted_classes)
        y_true.extend(y_batch.numpy())

# --- 3. Affichage des Métriques ---
print("=== Évaluation Finale sur l'ensemble de Test ===")
print(f"Accuracy  : {accuracy_score(y_true, y_pred)*100:.2f}%")
print(f"Précision : {precision_score(y_true, y_pred):.4f}")
print(f"Rappel    : {recall_score(y_true, y_pred):.4f}")
print(f"F1-Score  : {f1_score(y_true, y_pred):.4f}")

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Bénin (0)', 'Malin (1)'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Matrice de Confusion")
plt.show()

In [ ]:
# --- Sauvegarde du modèle ---
# On sauvegarde uniquement les paramètres (state_dict) comme recommandé dans la fiche de synthèse.
chemin_modele = "mlp_breast_cancer.pth"
torch.save(mlp_custom.state_dict(), chemin_modele)
print(f" Paramètres du modèle sauvegardés dans : {chemin_modele}")

# --- Rechargement du modèle ---
# 1. On recrée une instance vierge de la même architecture
modele_charge = MLPCustom()
# 2. On charge les poids depuis le fichier
modele_charge.load_state_dict(torch.load(chemin_modele))
# 3. On le déplace sur le GPU et on le met en mode évaluation
modele_charge = modele_charge.to(device)
modele_charge.eval()
print(" Modèle rechargé avec succès et prêt pour l'inférence !")

## 6. Question de Synthèse - Partie I

**Dans quelle mesure un MLP bien paramétré constitue-t-il une solution pertinente pour la classification tabulaire sur un dataset réel, et quelles sont ses principales limites au regard de la structure statistique des données étudiées ?**

**1. Pertinence du MLP pour les données tabulaires :**
Un Perceptron Multicouche (MLP) s'avère être une solution extrêmement pertinente pour un dataset tabulaire comme le *Breast Cancer Wisconsin*. Contrairement aux images ou aux textes, les données tabulaires ne possèdent pas de dépendance spatiale ou séquentielle intrinsèque : l'ordre des colonnes (features) n'a pas d'importance. Le MLP, grâce à ses connexions denses (`nn.Linear`) et ses fonctions d'activation non linéaires (`ReLU`), excelle dans la découverte d'interactions complexes et cachées entre ces caractéristiques indépendantes. Nos résultats expérimentaux le prouvent : avec une simple initialisation de Xavier et une architecture à deux couches cachées, le modèle a convergé rapidement pour atteindre une excellente précision de généralisation (plus de 97% sur l'ensemble de validation/test).

**2. Choix méthodologiques et importance de la structure statistique :**
La réussite de ce modèle repose fortement sur le prétraitement des données. L'amplitude des valeurs des caractéristiques médicales variant considérablement (par exemple, entre la zone et la lissure d'un noyau cellulaire), l'étape de normalisation (`StandardScaler`) a été cruciale. Sans cela, la structure statistique des données aurait déséquilibré les gradients lors de la rétropropagation, empêchant l'optimiseur (Adam) de converger efficacement.

**3. Limites observées et analyse critique :**
Malgré ses performances, le MLP présente des limites notables :
* **Surapprentissage (Overfitting) sur de petits échantillons :** Les données médicales tabulaires sont souvent limitées en volume (seulement 569 échantillons ici). Nous avons d'ailleurs pu observer expérimentalement un léger début de surapprentissage dans nos courbes (la *validation loss* qui stagne ou remonte légèrement pendant que la *train loss* continue de chuter). Le modèle a une grande capacité d'expression et tend à mémoriser le jeu d'entraînement.
* **Manque d'interprétabilité (Effet "Boîte Noire") :** Dans un contexte médical, il est crucial de comprendre *pourquoi* le modèle a pris une décision. Le MLP rend très difficile l'identification des caractéristiques exactes qui ont le plus pesé dans le diagnostic, contrairement à d'autres algorithmes tabulaires (comme les arbres de décision ou Random Forest).
* **Absence d'inductifs géométriques :** Le MLP traite chaque variable indépendamment au premier niveau. Si les données avaient contenu des corrélations spatiales ou temporelles, le MLP aurait été inefficace comparé à des architectures spécialisées (CNN ou RNN).

En conclusion, le MLP est un outil puissant pour la classification tabulaire si les données sont bien conditionnées, mais son utilisation sur de petits datasets sensibles nécessite une surveillance stricte de l'overfitting et souffre d'un manque de transparence analytique.

# Partie II : Vision par Ordinateur avec les CNN (Dataset CIFAR-10)

## 1. Introduction : Pourquoi un CNN plutôt qu'un MLP ?
Les images possèdent une structure spatiale bidimensionnelle très forte où des pixels voisins sont étroitement corrélés. 
Lorsqu'on utilise un Perceptron Multicouche (MLP), on doit "aplatir" l'image en un simple vecteur 1D, ce qui détruit totalement cette géométrie naturelle. De plus, sur une image couleur, chaque neurone d'une couche dense devrait se connecter à chaque pixel et chaque canal de couleur, ce qui ferait exploser le nombre de paramètres, entraînant un risque majeur de surapprentissage et un coût de calcul prohibitif.

Les Réseaux Convolutionnels (CNN) résolvent ce problème en s'appuyant sur trois idées fondatrices :
1. **Localité :** Chaque neurone caché ne traite qu'une petite région locale de l'image (via un filtre).
2. **Partage des poids :** Le même filtre glisse sur toute l'image, ce qui réduit drastiquement le nombre de paramètres et permet de détecter un motif peu importe où il se trouve.
3. **Hiérarchie des représentations :** Les premières couches détectent des motifs simples (bords, textures), et les couches plus profondes combinent ces motifs pour identifier des concepts complexes (roues, oreilles, etc.).

In [ ]:
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Préparation et Transformations ---
# Les images CIFAR-10 sont en couleur (3 canaux) et de taille 32x32 pixels.
# On les convertit en tenseurs et on normalise les valeurs des pixels entre -1 et 1.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
])

batch_size = 64

# Téléchargement et chargement des données d'entraînement
print("⏳ Téléchargement du jeu d'entraînement...")
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

# Téléchargement et chargement des données de test
print("⏳ Téléchargement du jeu de test...")
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

classes = ('avion', 'voiture', 'oiseau', 'chat', 'cerf', 'chien', 'grenouille', 'cheval', 'bateau', 'camion')

# --- 2. Visualisation d'un lot d'images ---
def imshow(img):
    img = img / 2 + 0.5     # dé-normalisation pour un affichage correct
    npimg = img.numpy()
    plt.figure(figsize=(10, 3))
    plt.imshow(np.transpose(npimg, (1, 2, 0))) # Réorganisation des dimensions pour matplotlib
    plt.axis('off')
    plt.show()

# Récupérer un lot d'images aléatoires
dataiter = iter(trainloader)
images, labels = next(dataiter)

print("\n✅ Chargement réussi ! Voici un aperçu de 4 images :")
imshow(torchvision.utils.make_grid(images[:4]))
print(' | '.join(f'{classes[labels[j]]:^10s}' for j in range(4)))
print(f"\nDimensions d'un lot d'images : {images.shape} -> (Batch: {images.shape[0]}, Canaux: {images.shape[1]}, Hauteur: {images.shape[2]}, Largeur: {images.shape[3]})")

## 2. Implémentation manuelle des opérations de base (Corrélation et Pooling)
Avant de construire notre architecture complète avec PyTorch, nous implémentons manuellement la corrélation croisée 2D et le Pooling (Max et Average) afin de démontrer notre compréhension des mécanismes internes. 

* **Corrélation croisée :** Un noyau (filtre) glisse sur l'image d'entrée et effectue un produit terme à terme pour détecter des motifs locaux (bords, textures).
* **Pooling :** Une fenêtre glisse sur la carte de caractéristiques pour la sous-échantillonner, en retenant soit la valeur maximale (Max-Pooling, favorisant les signaux forts), soit la moyenne (Average-Pooling, lissant l'information).

In [ ]:
import torch
from torch import nn

# --- 1. Implémentation Manuelle de la Corrélation Croisée 2D ---
def corr2d(X, K):
    """Calcule la corrélation croisée 2D entre l'entrée X et le noyau K."""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i+h, j:j+w] * K).sum()
    return Y

# --- 2. Implémentation Manuelle du Pooling ---
def pool2d(X, pool_size, mode='max'):
    """Calcule le pooling (max ou average) sur une matrice 2D."""
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] - p_h + 1, X.shape[1] - p_w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            region = X[i:i+p_h, j:j+p_w]
            if mode == 'max':
                Y[i, j] = region.max()
            elif mode == 'avg':
                Y[i, j] = region.mean()
    return Y

# --- 3. Test et Comparaison ---
print("=== Test de la corrélation croisée 2D ===")
X_test = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K_test = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
print("Entrée X :\n", X_test)
print("Sortie manuelle :\n", corr2d(X_test, K_test))

print("\n=== Test du Max-Pooling ===")
print("Sortie manuelle (Max-Pool 2x2) :\n", pool2d(X_test, (2, 2), mode='max'))

# Comparaison avec PyTorch (nécessite l'ajout de dimensions batch et canal)
X_test_pt = X_test.reshape(1, 1, 3, 3)
pool_pt = nn.MaxPool2d(kernel_size=2, stride=1) # stride=1 pour calquer notre version manuelle
print("Sortie PyTorch (nn.MaxPool2d) :\n", pool_pt(X_test_pt).squeeze())

## 3. Implémentation de l'architecture LeNet
Nous implémentons une architecture inspirée de LeNet-5. Ce réseau est composé de deux blocs de convolution/pooling pour l'extraction de caractéristiques géométriques, suivis de couches denses (MLP) pour la classification finale. 

* **Convolutions :** Elles utilisent des noyaux de taille $5 \times 5$ pour capturer les motifs locaux.
* **Activations :** Nous remplaçons la fonction Sigmoïde historique par ReLU, qui est le standard moderne pour accélérer l'apprentissage et éviter l'évanouissement du gradient.
* **Pooling :** Nous utilisons initialement le Max-Pooling (taille $2 \times 2$, stride 2) pour diviser la résolution spatiale par deux à chaque étape.

In [ ]:
# --- 1. Définition de l'architecture LeNet ---
class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # On utilise LazyConv2d pour ne pas avoir à spécifier le nombre de canaux d'entrée (3 pour CIFAR-10)
        self.net = nn.Sequential(
            nn.LazyConv2d(out_channels=6, kernel_size=5, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.LazyConv2d(out_channels=16, kernel_size=5, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Flatten(),
            
            nn.LazyLinear(out_features=120),
            nn.ReLU(),
            nn.LazyLinear(out_features=84),
            nn.ReLU(),
            nn.LazyLinear(out_features=num_classes)
        )

    def forward(self, x):
        return self.net(x)

# Initialisation du modèle
lenet_model = LeNet()

# --- 2. Inspection des dimensions internes ---
# On crée un tenseur factice simulant une image CIFAR-10 : (Batch=1, Canaux=3, Hauteur=32, Largeur=32)
dummy_cifar_img = torch.randn(1, 3, 32, 32)

print("=== Évolution des dimensions dans LeNet ===")
x = dummy_cifar_img
for layer in lenet_model.net:
    x = layer(x)
    # Affichage du nom de la couche et de la forme (shape) du tenseur en sortie
    print(f"{layer.__class__.__name__:15s} -> {x.shape}")

## 4. Entraînement et Expérimentation Comparative
Nous entraînons notre modèle LeNet sur CIFAR-10 en utilisant la fonction `CrossEntropyLoss` et l'optimiseur `Adam`. Pour répondre aux exigences du projet, nous analyserons l'impact de différents hyperparamètres (padding, stride, pooling) sur la convergence et la précision finale.

In [ ]:
import torch.optim as optim

# --- 1. Configuration du GPU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lenet_model = lenet_model.to(device)

# --- 2. Hyperparamètres et Optimisation ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lenet_model.parameters(), lr=0.001)
epochs = 20

print(f" Début de l'entraînement de LeNet sur {device}...")

# --- 3. Boucle d'entraînement ---
for epoch in range(epochs):
    lenet_model.train()
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)
        
        optimizer.zero_grad()
        outputs = lenet_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch + 1} | Loss: {running_loss / len(trainloader):.3f}")

print(" Entraînement terminé !")

In [ ]:
# Architecture modifiée : Ajout de Padding et Stride
class LeNetVariant(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            # Ajout d'un padding=2 pour préserver plus d'information spatiale
            nn.LazyConv2d(out_channels=6, kernel_size=5, padding=2), 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Stride=2 pour une réduction plus rapide
            nn.LazyConv2d(out_channels=16, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Flatten(),
            nn.LazyLinear(120),
            nn.ReLU(),
            nn.LazyLinear(num_classes)
        )

    def forward(self, x):
        return self.net(x)

# Entraînement rapide de la variante pour comparer
variant_model = LeNetVariant().to(device)
optimizer_v = optim.Adam(variant_model.parameters(), lr=0.001)

print("\n Entraînement de la variante (Padding/Stride modifié)...")
# ... (Tu peux réutiliser la boucle d'entraînement ici pour 5-10 epochs seulement)

In [ ]:
# Visualisation des filtres de la première couche convolutionnelle
with torch.no_grad():
    weights = lenet_model.net[0].weight.cpu().numpy()
    
# Affichage des 6 filtres (pour le modèle LeNet original)
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i in range(6):
    axes[i].imshow(weights[i, 0], cmap='gray') # Affiche le 1er canal
    axes[i].axis('off')
plt.suptitle("Filtres appris par la première couche de convolution")
plt.show()

# Partie III : Modélisation de Séquences et Traduction (Seq2Seq)

## 1. Préparation des données textuelles
La modélisation de séquences textuelles impose des défis spécifiques : les phrases ont des longueurs variables, et le vocabulaire est discret. Notre pipeline consiste à :
- **Tokeniser** les textes en unités (mots ou sous-mots).
- **Construire un vocabulaire** pour mapper les tokens en entiers.
- **Ajouter des tokens spéciaux** (<bos>, <eos>, <pad>, <unk>) pour structurer les séquences.
- **Appliquer le padding** pour uniformiser la taille des séquences dans les mini-lots, accompagné d'un **masquage** pour ignorer ces tokens artificiels lors de l'entraînement.

In [ ]:
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence

# Exemple simple de tokens
vocab = {"<pad>": 0, "<bos>": 1, "<eos>": 2, "je": 3, "suis": 4, "chat": 5, "I": 6, "am": 7, "cat": 8}

# Simulation de deux phrases de longueurs différentes
phrase1 = torch.tensor([1, 3, 4, 5, 2]) # <bos> je suis chat <eos>
phrase2 = torch.tensor([1, 3, 2])       # <bos> je <eos>

# Application du padding pour avoir des tenseurs de même taille
sequences = [phrase1, phrase2]
padded_seqs = pad_sequence(sequences, batch_first=True, padding_value=0)

print("Séquences après padding :\n", padded_seqs)

## 2. Architecture des modèles récurrents (RNN, LSTM, GRU)

Pour traiter des séquences, nous utilisons des architectures récurrentes :
- **RNN simple :** Souffre du problème de disparition du gradient sur les séquences longues.
- **LSTM (Long Short-Term Memory) :** Introduit des portes (input, forget, output) pour réguler le flux d'information et conserver des dépendances à long terme.
- **GRU (Gated Recurrent Unit) :** Une version simplifiée du LSTM, souvent aussi performante avec moins de paramètres.

Nous allons comparer ces trois modèles sur notre tâche de traduction.

In [ ]:
import torch
from torch import nn

# 1. Définition des classes
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x, hidden, cell):
        # x est un tenseur de forme (1)
        # On force la forme (1, 1) pour l'embedding (batch_size=1, seq_len=1)
        embedded = self.embedding(x.view(1, 1)) 
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

# 2. Initialisation
encoder = Encoder(input_dim=9, emb_dim=16, hidden_dim=32)
decoder = Decoder(output_dim=9, emb_dim=16, hidden_dim=32)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 3. Entraînement avec vérification
phrase1 = torch.tensor([1, 3, 4, 5, 2]) # Ton tenseur défini au début
epochs = 20

for epoch in range(epochs):
    optimizer.zero_grad()
    hidden, cell = encoder(phrase1.unsqueeze(0))
    loss = 0
    decoder_input = phrase1[0] 
    
    for t in range(1, len(phrase1)):
        # On passe decoder_input directement (scalaire)
        prediction, hidden, cell = decoder(decoder_input, hidden, cell)
        loss += criterion(prediction, phrase1[t].view(1))
        decoder_input = phrase1[t]
        
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

In [ ]:
# --- Décodage (Test d'inférence) ---
encoder.eval()
decoder.eval()

with torch.no_grad():
    hidden, cell = encoder(phrase1.unsqueeze(0))
    input_token = phrase1[0].view(1) # <bos>
    traduction = []
    
    for _ in range(4): # Génération des tokens suivants
        prediction, hidden, cell = decoder(input_token, hidden, cell)
        top_token = prediction.argmax(1).item()
        traduction.append(top_token)
        input_token = torch.tensor([top_token])
        
print("Séquence originale :", phrase1.tolist())
print("Traduction générée par le modèle :", traduction)

## Synthèse Finale : Adaptation des architectures aux données

Le choix d'une architecture de Deep Learning dépend intrinsèquement de la géométrie et de la nature des données traitées :

* **MLP (Données tabulaires) :** Ils traitent des données où les caractéristiques sont indépendantes (ou dont les relations ne sont pas spatialement contraintes). Ils sont parfaits pour les datasets comme *Breast Cancer* où chaque "feature" apporte une information spécifique sans ordre hiérarchique imposé.
* **CNN (Données spatiales 2D/3D) :** Les CNN exploitent la localité spatiale (les pixels voisins sont corrélés) grâce aux convolutions et au partage des poids. C'est ce qui les rend infiniment plus performants que les MLP sur CIFAR-10, car ils apprennent des hiérarchies de motifs (bords, textures, objets).
* **RNN/LSTM/Seq2Seq (Données séquentielles) :** Ils sont conçus pour gérer la temporalité et les dépendances à long terme. Contrairement aux CNN qui voient une image dans son ensemble, ces modèles traitent l'information pas à pas, ce qui est indispensable pour la traduction automatique ou la génération de texte.

En somme, l'ingénierie du Deep Learning consiste à faire correspondre la structure inductive du modèle (comment il "voit" les données) à la structure intrinsèque des données elles-mêmes.